# Chapter 6 — Heaps

*Built with [Claude](https://claude.ai) by Anthropic.*

Heaps give you O(log n) insert and O(1) peek at the minimum (or maximum). They are the engine behind top-K selection, priority queues, and streaming statistics.

In [10]:
import heapq
from typing import List

def show_heap(h, label='heap'):
    """Print heap contents with min at front."""
    print(f"  {label}: {h}  (min={h[0] if h else None})")

print('  heapq loaded (min-heap by default)')
print('  for max-heap: negate values or use (-val, val) tuples')

  heapq loaded (min-heap by default)
  for max-heap: negate values or use (-val, val) tuples
  heapq loaded (min-heap by default)
  for max-heap: negate values or use (-val, val) tuples


---

## Part 1 — Top-K Selection

**DS/ML connection:** `pd.Series.nlargest(k)`, `np.partition`, sklearn's `SelectKBest` — all top-K operations. Heaps are how you do this in a single pass over a stream without sorting everything.

In [11]:
# ── DS/ML PARALLEL: pandas nlargest / numpy partition ──
import pandas as pd
import numpy as np

scores = [3, 2, 1, 5, 6, 4]
k = 2
s = pd.Series(scores)

print(f"  scores:          {scores}")
print(f"  nlargest({k}):    {s.nlargest(k).tolist()}")
print(f"  np.partition:    {sorted(np.partition(scores, -k)[-k:], reverse=True)}")
print(f"  heapq.nlargest:  {heapq.nlargest(k, scores)}")

  scores:          [3, 2, 1, 5, 6, 4]
  nlargest(2):    [6, 5]
  np.partition:    [np.int64(6), np.int64(5)]
  heapq.nlargest:  [6, 5]
  scores:          [3, 2, 1, 5, 6, 4]
  nlargest(2):    [6, 5]
  np.partition:    [np.int64(6), np.int64(5)]
  heapq.nlargest:  [6, 5]


### LC 215 — [Kth Largest Element in an Array](https://leetcode.com/problems/kth-largest-element-in-an-array/)

**Problem:** Return the kth largest element (not kth distinct).

**DS parallel:** `pd.Series.nlargest(k).iloc[-1]` — finding the boundary value for a top-K cutoff.

**Key insight:** Maintain a min-heap of size k. Every element that exceeds the heap's minimum gets in; the heap minimum is always the kth largest seen so far.

In [12]:
# ── DS/ML APPROACH: pandas / numpy ──
def findKthLargest_pandas(nums, k):
    return pd.Series(nums).nlargest(k).iloc[-1]

def findKthLargest_partition(nums, k):
    return int(np.partition(nums, -k)[-k])

# ── RAW PYTHON (interview answer) ──
def findKthLargest(nums: List[int], k: int) -> int:
    heap = []   # min-heap of size k
    for num in nums:
        heapq.heappush(heap, num)
        if len(heap) > k:
            heapq.heappop(heap)   # evict smallest
    return heap[0]  # smallest of top-k = kth largest

nums = [3, 2, 1, 5, 6, 4]
k = 2
print(f"  nums={nums}, k={k}")
print(f"  pandas:    {findKthLargest_pandas(nums, k)}")
print(f"  partition: {findKthLargest_partition(nums, k)}")
print(f"  heap:      {findKthLargest(nums, k)}")

  nums=[3, 2, 1, 5, 6, 4], k=2
  pandas:    5
  partition: 5
  heap:      5
  nums=[3, 2, 1, 5, 6, 4], k=2
  pandas:    5
  partition: 5
  heap:      5


In [13]:
# ── TRACE: LC 215 ──
def findKthLargest_trace(nums, k):
    heap = []
    print(f"  nums={nums}, k={k}")
    print(f"  {'num':>5}  {'action':<20}  heap (min-heap of size≤{k})")
    for num in nums:
        heapq.heappush(heap, num)
        if len(heap) > k:
            evicted = heapq.heappop(heap)
            action = f"push {num}, evict {evicted}"
        else:
            action = f"push {num}"
        print(f"  {num:>5}  {action:<20}  {sorted(heap)}")
    print(f"  kth largest = heap[0] = {heap[0]}")

findKthLargest_trace([3, 2, 1, 5, 6, 4], 2)

  nums=[3, 2, 1, 5, 6, 4], k=2
    num  action                heap (min-heap of size≤2)
      3  push 3                [3]
      2  push 2                [2, 3]
      1  push 1, evict 1       [2, 3]
      5  push 5, evict 2       [3, 5]
      6  push 6, evict 3       [5, 6]
      4  push 4, evict 4       [5, 6]
  kth largest = heap[0] = 5
  nums=[3, 2, 1, 5, 6, 4], k=2
    num  action                heap (min-heap of size≤2)
      3  push 3                [3]
      2  push 2                [2, 3]
      1  push 1, evict 1       [2, 3]
      5  push 5, evict 2       [3, 5]
      6  push 6, evict 3       [5, 6]
      4  push 4, evict 4       [5, 6]
  kth largest = heap[0] = 5


### LC 347 — [Top K Frequent Elements](https://leetcode.com/problems/top-k-frequent-elements/)

**Problem:** Return the k most frequent elements.

**DS parallel:** `pd.Series.value_counts().nlargest(k)` — top-k by frequency for categorical feature analysis.

**Key insight:** Build a frequency map, then use a min-heap of size k keyed by frequency. Evict the least frequent when size exceeds k.

In [14]:
from collections import Counter

# ── DS/ML APPROACH: value_counts ──
def topKFrequent_pandas(nums, k):
    return pd.Series(nums).value_counts().nlargest(k).index.tolist()

# ── RAW PYTHON (interview answer) ──
def topKFrequent(nums: List[int], k: int) -> List[int]:
    freq = Counter(nums)
    # Min-heap: (count, element) — evict least frequent
    heap = []
    for elem, count in freq.items():
        heapq.heappush(heap, (count, elem))
        if len(heap) > k:
            heapq.heappop(heap)
    return [elem for _, elem in heap]

nums = [1, 1, 1, 2, 2, 3]
k = 2
print(f"  nums={nums}, k={k}")
print(f"  value_counts: {topKFrequent_pandas(nums, k)}")
print(f"  heap:         {topKFrequent(nums, k)}")

nums2 = [4,1,2,1,3,2,4,4]
print(f"  nums={nums2}, k=2")
print(f"  heap:         {topKFrequent(nums2, 2)}")

  nums=[1, 1, 1, 2, 2, 3], k=2
  value_counts: [1, 2]
  heap:         [2, 1]
  nums=[4, 1, 2, 1, 3, 2, 4, 4], k=2
  heap:         [2, 4]
  nums=[1, 1, 1, 2, 2, 3], k=2
  value_counts: [1, 2]
  heap:         [2, 1]
  nums=[4, 1, 2, 1, 3, 2, 4, 4], k=2
  heap:         [2, 4]


In [15]:
# ── TRACE: LC 347 ──
def topKFrequent_trace(nums, k):
    freq = Counter(nums)
    print(f"  freq: {dict(freq)}")
    heap = []
    print(f"  {'elem':>6}  {'count':>6}  {'action':<25}  heap")
    for elem, count in freq.items():
        heapq.heappush(heap, (count, elem))
        if len(heap) > k:
            evicted = heapq.heappop(heap)
            action = f"push ({count},{elem}), evict {evicted}"
        else:
            action = f"push ({count},{elem})"
        print(f"  {elem:>6}  {count:>6}  {action:<25}  {sorted(heap)}")
    result = [e for _, e in heap]
    print(f"  top {k} frequent: {result}")

topKFrequent_trace([1, 1, 1, 2, 2, 3], 2)

  freq: {1: 3, 2: 2, 3: 1}
    elem   count  action                     heap
       1       3  push (3,1)                 [(3, 1)]
       2       2  push (2,2)                 [(2, 2), (3, 1)]
       3       1  push (1,3), evict (1, 3)   [(2, 2), (3, 1)]
  top 2 frequent: [2, 1]
  freq: {1: 3, 2: 2, 3: 1}
    elem   count  action                     heap
       1       3  push (3,1)                 [(3, 1)]
       2       2  push (2,2)                 [(2, 2), (3, 1)]
       3       1  push (1,3), evict (1, 3)   [(2, 2), (3, 1)]
  top 2 frequent: [2, 1]


---

## Part 2 — Two-Heap Pattern (Streaming Median)

**DS/ML connection:** Online/streaming median computation — you can't store all values from an infinite stream. Two heaps maintain the lower and upper halves so the median is always O(1) to read.

In [16]:
# ── DS/ML PARALLEL: pandas rolling median ──
data = [1, 2, 3, 4, 5, 6, 7]
s = pd.Series(data)
print('  pandas rolling(3) median:')
print(' ', s.rolling(3, min_periods=1).median().tolist())
print('  (pandas buffers the window; two-heap is the streaming equivalent)')

  pandas rolling(3) median:
  [1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0]
  (pandas buffers the window; two-heap is the streaming equivalent)
  pandas rolling(3) median:
  [1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0]
  (pandas buffers the window; two-heap is the streaming equivalent)


### LC 295 — [Find Median from Data Stream](https://leetcode.com/problems/find-median-from-data-stream/)

**Problem:** Design a data structure that supports `addNum(num)` and `findMedian()` in O(log n) and O(1) respectively.

**DS parallel:** Online median tracking in a data stream — `pd.Series.expanding().median()` but without storing all values.

**Key insight:** Use two heaps: a max-heap for the lower half and a min-heap for the upper half. Keep them balanced (sizes differ by at most 1). The median is always at the top(s).

In [17]:
# ── DS/ML APPROACH: expanding median with pandas ──
def expanding_median(stream):
    result = []
    seen = []
    for x in stream:
        seen.append(x)
        result.append(pd.Series(seen).median())
    return result

# ── RAW PYTHON (interview answer) ──
class MedianFinder:
    def __init__(self):
        self.lo = []  # max-heap (negate for Python): lower half
        self.hi = []  # min-heap: upper half

    def addNum(self, num: int) -> None:
        # Always push to lo first
        heapq.heappush(self.lo, -num)
        # Balance: lo's max must be <= hi's min
        if self.hi and -self.lo[0] > self.hi[0]:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        # Rebalance sizes (lo can be at most 1 larger)
        if len(self.lo) > len(self.hi) + 1:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        elif len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def findMedian(self) -> float:
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])
        return (-self.lo[0] + self.hi[0]) / 2.0

stream = [1, 2, 3, 4, 5]
mf = MedianFinder()
medians_heap = []
for x in stream:
    mf.addNum(x)
    medians_heap.append(mf.findMedian())

print(f"  stream:          {stream}")
print(f"  pandas expanding: {expanding_median(stream)}")
print(f"  two-heap:         {medians_heap}")

  stream:          [1, 2, 3, 4, 5]
  pandas expanding: [np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0)]
  two-heap:         [1.0, 1.5, 2.0, 2.5, 3.0]
  stream:          [1, 2, 3, 4, 5]
  pandas expanding: [np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0)]
  two-heap:         [1.0, 1.5, 2.0, 2.5, 3.0]


In [18]:
# ── TRACE: LC 295 ──
class MedianFinder_trace:
    def __init__(self):
        self.lo = []
        self.hi = []

    def addNum(self, num):
        heapq.heappush(self.lo, -num)
        if self.hi and -self.lo[0] > self.hi[0]:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        if len(self.lo) > len(self.hi) + 1:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        elif len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))
        lo_vals = sorted([-x for x in self.lo], reverse=True)
        hi_vals = sorted(self.hi)
        median = self.findMedian()
        print(f"  add {num:>3}  lo(max-heap)={lo_vals}  hi(min-heap)={hi_vals}  median={median}")

    def findMedian(self):
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])
        return (-self.lo[0] + self.hi[0]) / 2.0

mft = MedianFinder_trace()
for x in [1, 2, 3, 4, 5, 6]:
    mft.addNum(x)

  add   1  lo(max-heap)=[1]  hi(min-heap)=[]  median=1.0
  add   2  lo(max-heap)=[1]  hi(min-heap)=[2]  median=1.5
  add   3  lo(max-heap)=[2, 1]  hi(min-heap)=[3]  median=2.0
  add   4  lo(max-heap)=[2, 1]  hi(min-heap)=[3, 4]  median=2.5
  add   5  lo(max-heap)=[3, 2, 1]  hi(min-heap)=[4, 5]  median=3.0
  add   6  lo(max-heap)=[3, 2, 1]  hi(min-heap)=[4, 5, 6]  median=3.5
  add   1  lo(max-heap)=[1]  hi(min-heap)=[]  median=1.0
  add   2  lo(max-heap)=[1]  hi(min-heap)=[2]  median=1.5
  add   3  lo(max-heap)=[2, 1]  hi(min-heap)=[3]  median=2.0
  add   4  lo(max-heap)=[2, 1]  hi(min-heap)=[3, 4]  median=2.5
  add   5  lo(max-heap)=[3, 2, 1]  hi(min-heap)=[4, 5]  median=3.0
  add   6  lo(max-heap)=[3, 2, 1]  hi(min-heap)=[4, 5, 6]  median=3.5
